In [1]:
from torchvision.datasets import MNIST 
from torch.utils.data import DataLoader 
from torchvision import transforms 
# use a transform to Normalise the images and convert to a tensor.
img_transform = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize((0.5, ), (0.5, )), # since MINIST dataset only has one channel 
])
# set the batch size (the number of images to use for each batch)
batch_size = 128
# load the data from the directory 'data' (and download it if not already downloaded)
dataset = MNIST('./data', transform=img_transform, download = True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [2]:
from torch import nn
from torch import tanh as tanh
from torch.nn import functional as F
class autoencoder(nn.Module):
    # we build the component of our nn. 
    def __init__(self):
        super(autoencoder, self).__init__()
        # MNISTimages are 28x28, single channel.
        # for nn.Conv2d(1, 16, 3, stride = 3, padding = 1) 
        # 1: the number of channels 
        # 16: the number of kernel
        # 3: the size of kernel: 3 * 3 
        # [size + 2 * padding - kernel] / stride + 1
        self.conv1 = nn.Conv2d(1, 16, 3, stride = 3, padding = 1) # b 16 10 10 
        # pooling layer: for shrinking down the size of resulted figures
        # [Size - kernel] / stride + 1 = (10 - 2) /2 + 1 = 5
        self.pool2 = nn.MaxPool2d(2, stride = 2) # b 16 ,5 ,5 
        self.conv2 = nn.Conv2d(16, 8, 3, stride = 2, padding =1) # b, 8, 2,2
        self.pool1 = nn.MaxPool2d(2, stride = 1) # b, 16, 2,2 
        # (size - 1) * Stride - 2 * Padding + Kernel + outputpadding = (2-1) *2 +  3 = 5
        self.iconv1 = nn.ConvTranspose2d(8,16,3, stride = 2) # b, 16, 5, 5 
        self.iconv2 = nn.ConvTranspose2d(16, 8 ,5, stride = 3, padding = 1) # 8, 15, 15
        self.iconv3 = nn.ConvTranspose2d(8,1,2, stride = 2, padding=1) # 1 * 28 * 28 
    
    def encoder(self, x):
        # x is a single MNIST image.
        # write the code to covert the image to a smaller
        # form (vector or image).
        x = self.pool2(F.relu(self.conv1(x)))
        x = self.pool1(F.relu(self.conv2(x)))
        return(x)
        
    def decoder(self, x):
        # x is the encoded form of the image
        # write the code to convert the encoded form back to the original
        # MNIST image size.
        x = F.relu(self.iconv1(x))
        # Relu : For x >= 0 : X , X < 0 : 0 
        x = F.relu(self.iconv2(x))
        x = tanh(self.iconv3(x))
        return(x)
        
    def forward(self, x):
        x = self.encoder(x) # latent space 
        x = self.decoder(x)
        return x

# Input: 1*28*28 -> 16* 10*10 -> 16* 5* 5 (after pool 2)->  8*3*3 -> 8*2*2 -(pool 1)>  Latent Space (8*2*2) -> 16*5*5 -> 8*15*15 -> 1*28*28 (End of decoder)

In [7]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torch import optim
from torchvision.utils import save_image
num_epochs = 100
learning_rate = 1e-3 
model = autoencoder().to(device)  # send our model from CPU to GPU! 
# The usage of your NN is determined by the LOSS function !! 
# MSELoss(y_true, y_predict)
criterion = nn.MSELoss() # nn.CrossEntropy() 
# weight_decay is for the learning rate 
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)  # Adam
for epoch in range(num_epochs):
# Iterate num_epoch number of times.
    for data in dataloader:
    # data contains the number of images specified in batch_size 128
    # this will loop until all 60000 MNIST images are provided to the AE
        img, _ = data
        img = img.to(device) # move our data from CPU to GPU! 
        # pass the images through the AE
        output = model(img)
        # compare the images to the output
        loss = criterion(output, img) # our task is recontruction ! 
        # update the gradients to minimise the loss
        optimizer.zero_grad() 
        loss.backward()
        optimizer.step()
    print('epoch [{}/{}], loss:{}'.format(epoch+1, num_epochs, loss.item()))
    # same some of the images to examine the improvement in the AE
    if epoch % 10 == 0:
        imageGrid = to_img(output.data)
        save_image(imageGrid, './imageGrid_{}.png'.format(epoch))

epoch [1/100], loss:0.22946378588676453
epoch [2/100], loss:0.17932774126529694
epoch [3/100], loss:0.15694639086723328
epoch [4/100], loss:0.14578652381896973
epoch [5/100], loss:0.135880246758461
epoch [6/100], loss:0.12609553337097168
epoch [7/100], loss:0.11968765407800674
epoch [8/100], loss:0.129352405667305
epoch [9/100], loss:0.12831470370292664
epoch [10/100], loss:0.11559425294399261
epoch [11/100], loss:0.12334857136011124
epoch [12/100], loss:0.10637033730745316
epoch [13/100], loss:0.11795733869075775
epoch [14/100], loss:0.12091419845819473
epoch [15/100], loss:0.12020427733659744
epoch [16/100], loss:0.11289245635271072
epoch [17/100], loss:0.11264043301343918
epoch [18/100], loss:0.11524976044893265
epoch [19/100], loss:0.11617421358823776
epoch [20/100], loss:0.11250930279493332
epoch [21/100], loss:0.10699721425771713
epoch [22/100], loss:0.10202328115701675
epoch [23/100], loss:0.11809304356575012
epoch [24/100], loss:0.10913518071174622
epoch [25/100], loss:0.104215

In [5]:
from torch import save as torch_save
torch_save(model.state_dict(), './conv_autoencoder.pth')

In [4]:
def to_img(x):
    x = 0.5 * (x + 1)
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 1, 28, 28)
    return x
## output comes from the decoder
imageGrid = to_img(output.data)
save_image(imageGrid, './image_{}.png'.format(epoch))

In [6]:
from torch import randn as torch_randn
# img is an image batch from the data loader
noise = torch_randn(img.size()) * 0.2